# Teach an image model to predict survival

A small CNN learns survival curves from handwritten digits, with half the outcomes censored and without seeing the digit labels. Synthetic outcomes let us compare its predictions with the truth.

From the repository root, launch with:
```bash
uv run --group docs --group examples jupyter lab examples/survival_mnist/survival-mnist.ipynb
```
Choose **Python 3**, then **Restart Kernel and Run All Cells**. Run inside this checkout; the setup finds the repository from either its root or the notebook directory. Use a fresh kernel with JAX's default float32. The first run downloads MNIST into `~/.cache/survjax-examples/mnist`; set `SURVJAX_EXAMPLES_DATA` before launch to use another cache.

`CONFIG = "smoke"` trains for one epoch on 5,000 images, selects on 1,000 validation images and evaluates 2,000 test images. Change it to `"full"` and restart/run all for ten epochs on 50,000/10,000/10,000 images. Figures go into a new temporary directory; committed results stay unchanged.

In [ ]:
import csv
import inspect
import sys
import tempfile
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
from IPython.display import SVG, Code, Markdown, display

import survjax

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT))

CONFIG = "smoke"
OUT = Path(tempfile.mkdtemp(prefix="survjax-mnist-notebook-"))
HEADER = "| Model | Mean log likelihood | Shape k | Harrell's C |\n| --- | ---: | ---: | ---: |\n"

## A world where we know the answer

Digit $d$ determines scale through $\log b_d=\log 10-0.2d$, with shared shape $k=1.38$. Zeros tend to survive longer than nines. Censoring is Uniform$(0,7)$, so about half the event times are hidden. Labels generate outcomes and supply the oracle; they never enter training.

In [ ]:
from examples.survival_mnist import data, model
from examples.survival_mnist.run import _figures

pixels, labels = data.load_mnist()
outcomes = data.generate_outcomes(labels)
observations = survjax.SurvivalData(outcomes.time, outcomes.event)
cohorts = data.split_indices(CONFIG)
training, validation, test = (cohorts[name] for name in ("train", "validation", "test"))
print({name: len(indices) for name, indices in cohorts.items()})

times = jnp.linspace(0, 20, 201, dtype=jnp.float32)
oracle = survjax.Weibull.from_log(
    log_shape=jnp.log(data.TRUE_SHAPE),
    log_scale=data.TRUE_LOG_SCALES,
)
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
ax.plot(times, oracle.survival_grid(times), label=[str(d) for d in range(10)])
ax.set(xlabel="Time", ylabel="True survival", ylim=(0, 1))
ax.legend(title="Digit", ncols=5)
plt.show()
plt.close(fig)

## Your network, a survival likelihood

The CNN maps an image to a log scale. A shared learned log shape completes the Weibull model. The loss connects the network to survjax:
```python
def loss(params, images, data):
    cnn, log_k = params
    dist = survjax.Weibull.from_log(log_shape=log_k, log_scale=jax.vmap(cnn)(images))
    return -dist.log_likelihood(data).mean()
```
The next cell shows the CNN source and trains using the functions in [model.py](model.py). Adam updates the network and shape in batches of 128 at learning rate 0.001; `data[idx]` keeps observations aligned with images. Validation selects the checkpoint. BFGS fits the two shared intercept parameters, providing initialization and a baseline without image information.

To change the network or loss, edit `model.py`, then restart the kernel and run all cells.

In [ ]:
display(Code(inspect.getsource(model.CNN), language="python"))
intercept = model.fit_intercept(outcomes.time[training], outcomes.event[training])
result = model.train(
    pixels[training],
    observations[training],
    pixels[validation],
    observations[validation],
    initial_intercept=intercept,
    epochs=1 if CONFIG == "smoke" else 10,
)
cnn, log_k = result.params
log_scales = model.predict(cnn, pixels[test])
print(f"Selected epoch: {result.selected_epoch}")

## Evaluate and inspect your run

Higher held-out log likelihood and Harrell's C are better. The intercept baseline has no image information; the oracle knows each image's generating parameters. A single smoke epoch exercises the pipeline; use full training to see the learned structure.

The figures show predicted and true curves for one image per digit, the distribution of predicted log scales, and epoch training/validation loss. `cnn`, `log_k`, `log_scales` and `result` remain available for exploration.

In [ ]:
rows = [
    dict(model=name, **model.evaluate(shape, scales, observations[test]))
    for name, shape, scales in (
        ("cnn", log_k, log_scales),
        ("intercept", *intercept),
        ("oracle", jnp.log(data.TRUE_SHAPE), data.TRUE_LOG_SCALES[labels[test]]),
    )
]
display(
    Markdown(
        HEADER
        + "".join(
            f"| {r['model']} | {r['mean_log_likelihood']:.3f} | {r['shape']:.3f} | {r['concordance']:.3f} |\n"
            for r in rows
        )
    )
)
_figures(pixels[test], labels[test], log_k, log_scales, result, OUT)
for name in ("fig1.svg", "fig2.svg", "fig3.svg"):
    display(SVG(filename=str(OUT / name)))
print(f"Your figures: {OUT}")

## Compare with the committed full experiment

These saved results use 50,000 training, 10,000 validation and 10,000 official test images over ten epochs. They are distinct from the run above. Validation selected epoch 7; the learned shape is 1.368 compared with generating shape 1.380. Full training beats the intercept baseline on both metrics.

In [ ]:
FULL = ROOT / "examples/survival_mnist/results"
with (FULL / "metrics.csv").open() as stream:
    full_rows = list(csv.DictReader(stream))
display(
    Markdown(
        HEADER
        + "".join(
            f"| {r['model']} | {float(r['mean_log_likelihood']):.3f} | {float(r['shape']):.3f} | {float(r['concordance']):.3f} |\n"
            for r in full_rows
        )
    )
)
display(SVG(filename=str(FULL / "fig1.svg")))

## Bring your own features

Swap the CNN for your own network or tabular features. A family's `from_log` connects network outputs to survival parameters; minibatches remain ordinary `data[idx]` slices. [Likelihood](../../docs/likelihood.md) and [concordance](../../docs/concordance.md) come from the same library.

We chose a shared shape and per-image scale. A network could instead predict both log shape and log scale per image. With different shapes, curves can cross, so choose a risk score suited to the prediction you want to compare.

See the [walkthrough](../../docs/use-cases/survival-mnist.md) and [command-line runner](README.md) for more context and saved CSV/metadata outputs.

MNIST is by Y. LeCun, C. Cortes and C. J. C. Burges, under CC BY-SA 3.0.
[Dataset](http://yann.lecun.com/exdb/mnist/); survival outcomes are synthetic.